# CivicPulse Qwen3-VL Colab API bridge

Run this notebook in Google Colab with a GPU runtime. It loads Qwen3-VL-4B-Instruct, exposes an authenticated OpenAI-compatible image endpoint, and creates a temporary Cloudflare HTTPS tunnel for CivicPulse.

**Prototype only:** Colab sessions can stop and free GPU availability is not guaranteed. Use synthetic, public, or consented test images only. Keep the generated token private. Human review remains mandatory.


## 1. Enable GPU

In Colab select **Runtime → Change runtime type → T4 GPU** (if available). Run cells from top to bottom.

In [ ]:
!pip -q install -U "transformers>=4.57.0" accelerate bitsandbytes qwen-vl-utils fastapi uvicorn python-multipart pillow requests


In [ ]:
import torch, sys
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Select Runtime > Change runtime type > T4 GPU, if available.")
print("GPU:", torch.cuda.get_device_name(0))


## 2. Download the integration runner from GitHub

This downloads the API implementation from the `rebuild/architecture-blueprint` branch. No API token is stored in this repository.

In [ ]:
!wget -q https://raw.githubusercontent.com/ankushdixit0001/CivicPulse-AI-Production/rebuild/architecture-blueprint/ai-vision-service/notebooks/civicpulse_qwen_colab_api.py -O civicpulse_qwen_colab_api.py
!python -m py_compile civicpulse_qwen_colab_api.py
print("Downloaded and syntax-checked CivicPulse Qwen API runner.")


## 3. Start Qwen API and temporary HTTPS tunnel

The next cell loads the 4-bit model, starts the authenticated `/v1/chat/completions` endpoint, and opens a temporary Cloudflare quick tunnel. It prints a one-time API token and a base URL. Copy both privately into Render environment variables. **Never paste the token into GitHub, frontend code, or chat.**

In [ ]:
%run civicpulse_qwen_colab_api.py


## 4. Configure Render manually

In the Render environment for `civicpulse-ai-vision`, set:

- `QWEN_VL_BASE_URL`: printed Cloudflare URL ending in `/v1`
- `QWEN_VL_API_KEY`: printed `CIVICPULSE_QWEN_TOKEN`
- `QWEN_VL_MODEL`: `Qwen/Qwen3-VL-4B-Instruct`
- `AI_PROVIDER_PRIORITY`: `qwen_vl`
- `ALLOW_EXTERNAL_IMAGE_AI`: `true` only during controlled testing

Keep text and OpenRouter fallback flags disabled. Preserve existing `AI_SERVICE_TOKEN` for Supabase-to-Render authentication. `configured=true` in `/health/providers` is not proof of successful inference; test an image next.


## Limitations

- Colab and Cloudflare quick tunnels are temporary and not production-available 24/7.
- If Colab restarts, rerun this notebook and update the Render URL/token.
- The model may return an empty `bounding_boxes` array when it cannot localize a defect; do not fabricate boxes.
- Evaluate on labelled images and retain municipal human review before any production use.
